# Two Feet + Head IMU Stride Analysis

This notebook demonstrates gait analysis using multiple synchronized IMU sensors: left foot, right foot, head, and hand. It loads APDM `.h5` recordings, time-synchronizes them to a common overlapping window, detects walking bouts, runs dual-foot inertial mechanization with zero-velocity updates, and compares left/right stride metrics alongside head motion.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremydwong/stride_estimation_imu/blob/main/notebooks/demo_colab_two_feet_head_exphand.ipynb)

In [ ]:
from datetime import time as dt_time

# =============================================================================
# DATA SOURCE: set to 'demo' to use included sample data, or 'upload' to
# upload your own .h5 files in the next cell.
# =============================================================================
DATA_SOURCE = 'demo'  # 'demo' or 'upload'

# =============================================================================
# WALKING BOUT TARGETS (times in the recording's local timezone)
# Each tuple: (name, target_time, search_window_seconds)
#
# Timezone note: APDM timestamps are UTC microseconds since epoch. The sensor
# config stores the recording timezone, so the module automatically converts
# timestamps to the recording's local time. The target times below should
# match the clock time where/when the data was recorded.
# =============================================================================
WALKING_BOUT_TARGETS = [
    ('track walk', dt_time(16, 34, 0), 240),  # 4:34 PM local recording time
]

# =============================================================================
# BOUT DETECTION PARAMETERS
# =============================================================================
MIN_QUIET_SECONDS = 1.5   # Minimum quiet period to count as walk boundary
MIN_WALK_SECONDS = 3.0    # Minimum walking duration to include
W_THRESHOLD = 30.0        # Max angular velocity (deg/s) for quiet detection
A_THRESHOLD = 1.0         # Max accel deviation from gravity (m/s^2) for quiet

## Setup

Clone the repository and install the required Python packages. If you are running locally (not in Colab), you can skip this cell and install dependencies via `pip install -r requirements.txt`.

In [ ]:
# Clone repository (or pull latest) and install dependencies
import os
if os.path.isdir('stride_estimation_imu'):
    !cd stride_estimation_imu && git pull
else:
    !git clone https://github.com/jeremydwong/stride_estimation_imu.git
%cd stride_estimation_imu
!pip install -q numpy scipy matplotlib h5py

In [ ]:
import sys
import os
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

import stride_imu as imu
from stride_imu.apdm import load_imu_recording, find_overlapping_recordings, ImuRecording
from stride_imu.inertial import detect_walking_bouts, find_bouts_near_time, WalkingBout

print("stride_imu loaded successfully!")

## Load Data

Load APDM `.h5` sensor files for each body location. If `DATA_SOURCE = 'upload'`, you can upload files interactively (Colab only). If `DATA_SOURCE = 'demo'`, it uses the included sample data. This demo expects three separate sensor files: left foot, right foot, and head.

In [ ]:
if DATA_SOURCE == 'upload':
    try:
        from google.colab import files
        print("Select your APDM .h5 files to upload (left foot, right foot, head):")
        uploaded = files.upload()
        uploaded_files = sorted(uploaded.keys())
        print(f"\nUploaded {len(uploaded_files)} file(s): {uploaded_files}")
        if len(uploaded_files) >= 3:
            LEFT_FILE = uploaded_files[0]
            RIGHT_FILE = uploaded_files[1]
            HEAD_FILE = uploaded_files[2]
            HAND_FILE = HEAD_FILE
            print(f"\nAssigned:")
            print(f"  LEFT_FILE  = {LEFT_FILE}")
            print(f"  RIGHT_FILE = {RIGHT_FILE}")
            print(f"  HEAD_FILE  = {HEAD_FILE}")
            print("\nReview assignments above. If wrong, re-assign LEFT_FILE, RIGHT_FILE,")
            print("HEAD_FILE manually using the uploaded filenames.")
        else:
            print(f"\nExpected 3 files (left, right, head). Got {len(uploaded_files)}.")
            print("Assign LEFT_FILE, RIGHT_FILE, HEAD_FILE manually from:", uploaded_files)
            LEFT_FILE = uploaded_files[0] if len(uploaded_files) > 0 else ''
            RIGHT_FILE = uploaded_files[1] if len(uploaded_files) > 1 else ''
            HEAD_FILE = uploaded_files[2] if len(uploaded_files) > 2 else LEFT_FILE
            HAND_FILE = HEAD_FILE
    except ImportError:
        print("Not running in Colab. Set LEFT_FILE, RIGHT_FILE, HEAD_FILE manually.")
        LEFT_FILE = ''
        RIGHT_FILE = ''
        HEAD_FILE = ''
        HAND_FILE = ''
else:
    LEFT_FILE = 'data/20251029-154305_LF_Pilot_Ch_Oct29.h5'
    RIGHT_FILE = 'data/20251029-154310_RF_Pilot_Ch_Oct29.h5'
    HEAD_FILE = 'data/20251029-154313_Head_Pilot_Ch_Oct29.h5'
    HAND_FILE = HEAD_FILE
    print(f"Using demo data:")
    print(f"  LEFT_FILE  = {LEFT_FILE}")
    print(f"  RIGHT_FILE = {RIGHT_FILE}")
    print(f"  HEAD_FILE  = {HEAD_FILE}")

## Helper Functions

**You don't need to read the next two cells.** They contain simple plotting and analysis utility functions used later in the notebook. They're collapsed by default — expand them if you're curious, but they aren't important for understanding the gait analysis pipeline.

In [ ]:
#@title Helper functions (click to expand)
class BoutSelection:
    """A selected walking bout with its timing metadata.

    Wraps a detected WalkingBout with human-readable name and datetime info,
    used to pass bout context through the processing pipeline.
    """
    def __init__(self, name, bout, start_datetime, end_datetime, confirmed=False):
        self.name = name
        self.bout = bout
        self.start_datetime = start_datetime
        self.end_datetime = end_datetime
        self.start_idx = bout.start_idx
        self.end_idx = bout.end_idx
        self.duration_seconds = bout.duration_seconds
        self.confirmed = confirmed

    def __repr__(self):
        return (f"BoutSelection('{self.name}', "
                f"start={self.start_datetime.strftime('%H:%M:%S')}, "
                f"duration={self.duration_seconds:.1f}s)")


def compute_alignment_rotation(P):
    """Compute a 3x3 rotation matrix that aligns a trajectory's overall
    walking direction with the +Y axis, so that "forward" points up in plots.

    Parameters
    ----------
    P : ndarray, shape (N, 3)
        Position trajectory (X, Y, Z).

    Returns
    -------
    R : ndarray, shape (3, 3)
        Rotation matrix (Z-axis rotation only; Z values are unchanged).
    """
    direction = P[-1, :2] - P[0, :2]
    theta = np.arctan2(direction[1], direction[0])
    rotation_angle = np.pi / 2 - theta
    cos_a, sin_a = np.cos(rotation_angle), np.sin(rotation_angle)
    return np.array([[cos_a, -sin_a, 0], [sin_a, cos_a, 0], [0, 0, 1]])


def apply_rotation_and_offset(P, R, offset):
    """Rotate a trajectory about its starting point, then shift by an offset.

    Used to place left and right foot trajectories side by side in plots.

    Parameters
    ----------
    P : ndarray, shape (N, 3)
        Position trajectory.
    R : ndarray, shape (3, 3)
        Rotation matrix.
    offset : ndarray, shape (3,)
        Translation applied after rotation.

    Returns
    -------
    P_transformed : ndarray, shape (N, 3)
    """
    P_centered = P - P[0, :]
    P_rotated = (R @ P_centered.T).T
    return P_rotated + offset


def compute_total_distance(P):
    """Compute the total path length (sum of step-to-step distances).

    Parameters
    ----------
    P : ndarray, shape (N, 3)
        Position trajectory.

    Returns
    -------
    distance : float
        Total distance in meters.
    """
    diffs = np.diff(P, axis=0)
    return float(np.sum(np.sqrt(np.sum(diffs ** 2, axis=1))))


def compute_step_metrics(strides):
    """Extract summary statistics from stride segmentation output.

    Takes the dict returned by ``imu.stride_segmentation()`` and computes
    mean/std of step speed, step length, and step duration.

    Parameters
    ----------
    strides : dict
        Output of ``imu.stride_segmentation()``, containing keys
        ``'frwd_speed'``, ``'time'``, and ``'frwd'``.

    Returns
    -------
    metrics : dict
        Dictionary with keys: ``step_lengths``, ``step_durations``,
        ``step_speeds``, ``mean_speed``, ``std_speed``, ``mean_length``,
        ``std_length``, ``mean_duration``, ``std_duration``, ``n_steps``.
    """
    step_speeds = strides['frwd_speed']
    step_durations = strides['time']
    
    if len(step_speeds) == 0:
        return {'step_lengths': np.array([]), 'step_durations': np.array([]),
                'step_speeds': np.array([]), 'mean_speed': 0.0, 'std_speed': 0.0,
                'mean_length': 0.0, 'std_length': 0.0, 'mean_duration': 0.0,
                'std_duration': 0.0, 'n_steps': 0}
    
    step_lengths = strides['frwd'][-1, :]
    return {
        'step_lengths': step_lengths, 'step_durations': step_durations,
        'step_speeds': step_speeds,
        'mean_speed': np.mean(step_speeds), 'std_speed': np.std(step_speeds),
        'mean_length': np.mean(step_lengths), 'std_length': np.std(step_lengths),
        'mean_duration': np.mean(step_durations), 'std_duration': np.std(step_durations),
        'n_steps': len(step_speeds)
    }


def analyze_head_motion(head_recording, period):
    """Compute summary statistics of head IMU motion during a bout.

    Parameters
    ----------
    head_recording : ImuRecording
        Head sensor recording (sliced to bout range).
    period : float
        Sampling period in seconds.

    Returns
    -------
    metrics : dict
        Angular velocity and acceleration statistics (mean, std, max),
        plus raw time series ``'Wm'`` and ``'Am'``.
    """
    Wm = np.sqrt(np.sum(head_recording.Wb ** 2, axis=1)) / period * 180 / np.pi
    Am = np.sqrt(np.sum(head_recording.Ab ** 2, axis=1))
    return {
        'ang_vel_mean': np.mean(Wm), 'ang_vel_std': np.std(Wm), 'ang_vel_max': np.max(Wm),
        'accel_mean': np.mean(Am), 'accel_std': np.std(Am), 'accel_max': np.max(Am),
        'Wm': Wm, 'Am': Am
    }


def process_walking_bout(bout, left_rec, right_rec, head_rec, period):
    """Run the full dual-foot stride estimation pipeline on one walking bout.

    Slices all three recordings to the bout's time range, runs dual-foot
    inertial mechanization, segments strides for each foot, and computes
    head motion statistics.

    Parameters
    ----------
    bout : WalkingBout
        Detected walking bout with start/end indices.
    left_rec, right_rec, head_rec : ImuRecording
        Synchronized IMU recordings for each sensor.
    period : float
        Sampling period in seconds.

    Returns
    -------
    result : dict
        Contains ``'left_metrics'``, ``'right_metrics'``,
        ``'head_metrics'``, stride data, walk info, and total distances.
    """
    left_bout = left_rec[bout.start_idx:bout.end_idx]
    right_bout = right_rec[bout.start_idx:bout.end_idx]
    head_bout = head_rec[bout.start_idx:bout.end_idx]
    
    left_walk_info, right_walk_info = imu.compute_position_two_imus(
        left_bout.Wb, left_bout.Ab, right_bout.Wb, right_bout.Ab, period)
    
    left_strides = imu.stride_segmentation(left_walk_info, period)
    right_strides = imu.stride_segmentation(right_walk_info, period)
    
    return {
        'left_metrics': compute_step_metrics(left_strides),
        'right_metrics': compute_step_metrics(right_strides),
        'head_metrics': analyze_head_motion(head_bout, period),
        'left_strides': left_strides, 'right_strides': right_strides,
        'left_walk_info': left_walk_info, 'right_walk_info': right_walk_info,
        'left_bout': left_bout, 'right_bout': right_bout, 'head_bout': head_bout,
        'bout': bout,
        'left_total_distance': compute_total_distance(left_walk_info['P']),
        'right_total_distance': compute_total_distance(right_walk_info['P'])
    }

## Plotting Functions

**You don't need to read this cell either.** These are standard matplotlib plotting functions for visualizing trajectories and step metrics.

In [ ]:
#@title Plotting functions (click to expand)
def plot_rotation_corrected_trajectories(left_walk_info, right_walk_info, title="", dx_offset=0.3):
    """Plot left and right foot trajectories side by side, aligned to the
    forward walking direction.

    Each foot's trajectory is independently rotated so that forward points
    up (+Y), then offset horizontally so the two feet don't overlap.

    Parameters
    ----------
    left_walk_info, right_walk_info : dict
        Output of ``imu.compute_position_two_imus()``, each containing ``'P'``.
    title : str, optional
        Title suffix for the figure.
    dx_offset : float, optional
        Horizontal separation between feet in meters (default 0.3).

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    P_left, P_right = left_walk_info['P'], right_walk_info['P']
    R_left, R_right = compute_alignment_rotation(P_left), compute_alignment_rotation(P_right)
    P_left_rot = apply_rotation_and_offset(P_left, R_left, np.array([-dx_offset/2, 0, 0]))
    P_right_rot = apply_rotation_and_offset(P_right, R_right, np.array([dx_offset/2, 0, 0]))
    
    fig = plt.figure(figsize=(14, 6))
    fig.suptitle(f'Rotation Corrected Trajectories - {title}', fontsize=12)
    
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.plot(P_left_rot[:, 0], P_left_rot[:, 1], 'b-', linewidth=1, label='Left Foot', alpha=0.8)
    ax1.plot(P_right_rot[:, 0], P_right_rot[:, 1], 'r-', linewidth=1, label='Right Foot', alpha=0.8)
    ax1.plot(P_left_rot[0, 0], P_left_rot[0, 1], 'bo', markersize=8)
    ax1.plot(P_right_rot[0, 0], P_right_rot[0, 1], 'ro', markersize=8)
    ax1.set_xlabel('X (lateral) [m]'); ax1.set_ylabel('Y (forward) [m]')
    ax1.set_title('XY View'); ax1.legend(); ax1.grid(True, alpha=0.3); ax1.axis('equal')
    
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    ax2.plot(P_left_rot[:, 0], P_left_rot[:, 1], P_left_rot[:, 2], 'b-', linewidth=1, alpha=0.8)
    ax2.plot(P_right_rot[:, 0], P_right_rot[:, 1], P_right_rot[:, 2], 'r-', linewidth=1, alpha=0.8)
    ax2.set_xlabel('X [m]'); ax2.set_ylabel('Y [m]'); ax2.set_zlabel('Z [m]')
    ax2.set_title('3D View')
    plt.tight_layout()
    return fig


def plot_bout_summary(bout_name, left_metrics, right_metrics, head_metrics):
    """Create a 4-panel summary figure comparing left/right gait and head motion.

    Panels: (1) bar chart of mean step speed by foot, (2) bar chart of mean
    step length by foot, (3) overlaid step speed histograms, (4) text summary
    of head motion statistics and step counts.

    Parameters
    ----------
    bout_name : str
        Name of the walking bout (used in the title).
    left_metrics, right_metrics : dict
        Output of ``compute_step_metrics()`` for each foot.
    head_metrics : dict
        Output of ``analyze_head_motion()``.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'Walking Bout: {bout_name}', fontsize=14)
    
    x = np.arange(2)
    
    # Step speed
    ax = axes[0, 0]
    means = [left_metrics['mean_speed'], right_metrics['mean_speed']]
    stds = [left_metrics['std_speed'], right_metrics['std_speed']]
    ax.bar(x, means, yerr=stds, capsize=5, color=['blue', 'red'], alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(['Left', 'Right'])
    ax.set_ylabel('Step Speed [m/s]'); ax.set_title('Mean Step Speed'); ax.grid(True, axis='y')
    
    # Step length
    ax = axes[0, 1]
    means = [left_metrics['mean_length'], right_metrics['mean_length']]
    stds = [left_metrics['std_length'], right_metrics['std_length']]
    ax.bar(x, means, yerr=stds, capsize=5, color=['blue', 'red'], alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(['Left', 'Right'])
    ax.set_ylabel('Step Length [m]'); ax.set_title('Mean Step Length'); ax.grid(True, axis='y')
    
    # Step speed distribution
    ax = axes[1, 0]
    all_speeds = []
    if left_metrics['n_steps'] > 0: all_speeds.extend(left_metrics['step_speeds'])
    if right_metrics['n_steps'] > 0: all_speeds.extend(right_metrics['step_speeds'])
    if all_speeds:
        bins = np.linspace(min(all_speeds), max(all_speeds), 46)
        if left_metrics['n_steps'] > 0:
            ax.hist(left_metrics['step_speeds'], bins=bins, alpha=0.5, label='Left', color='blue')
        if right_metrics['n_steps'] > 0:
            ax.hist(right_metrics['step_speeds'], bins=bins, alpha=0.5, label='Right', color='red')
    ax.set_xlabel('Step Speed [m/s]'); ax.set_ylabel('Count')
    ax.set_title('Step Speed Distribution'); ax.legend(); ax.grid(True)
    
    # Summary text
    ax = axes[1, 1]
    text = (f"Head Motion Summary:\n"
            f"Angular Velocity:\n  Mean: {head_metrics['ang_vel_mean']:.1f} deg/s\n"
            f"  Std: {head_metrics['ang_vel_std']:.1f} deg/s\n  Max: {head_metrics['ang_vel_max']:.1f} deg/s\n\n"
            f"Acceleration:\n  Mean: {head_metrics['accel_mean']:.2f} m/s\u00b2\n"
            f"  Std: {head_metrics['accel_std']:.2f} m/s\u00b2\n  Max: {head_metrics['accel_max']:.2f} m/s\u00b2\n\n"
            f"Steps: L={left_metrics['n_steps']}, R={right_metrics['n_steps']}")
    ax.text(0.1, 0.5, text, transform=ax.transAxes, fontsize=11, verticalalignment='center', fontfamily='monospace')
    ax.axis('off'); ax.set_title('Head Motion & Step Counts')
    
    plt.tight_layout()
    return fig

## 1. Load and Synchronize IMU Recordings

Use `load_imu_recording()` to open each APDM `.h5` file, then call `find_overlapping_recordings()` to time-synchronize all sensors. Each sensor started recording at a slightly different time, so this function finds the common overlapping window and trims all recordings to the same time range using their raw APDM timestamps.

In [ ]:
print("Loading IMU recordings...")
left_rec = load_imu_recording(LEFT_FILE)
right_rec = load_imu_recording(RIGHT_FILE)
head_rec = load_imu_recording(HEAD_FILE)
hand_rec = load_imu_recording(HAND_FILE)

print(f"  Left foot: {len(left_rec)} samples")
print(f"  Right foot: {len(right_rec)} samples")
print(f"  Head: {len(head_rec)} samples")
print(f"  Hand: {len(hand_rec)} samples")
if left_rec.tz_offset_hours is not None:
    print(f"  Recording timezone: UTC{left_rec.tz_offset_hours:+.0f}")

print("\nSynchronizing recordings...")
synced = find_overlapping_recordings([left_rec, right_rec, head_rec, hand_rec])
left_synced, right_synced, head_synced, hand_synced = synced
PERIOD = left_synced.period

print(f"Sampling period: {PERIOD:.6f} s ({1/PERIOD:.1f} Hz)")
print(f"Synchronized samples: {len(left_synced)}")

## 2. Detect Walking Bouts

Scan the synchronized recording for walking bouts using the left foot signal. A walking bout is a period of sustained movement bounded on both sides by quiet (stationary) periods. The algorithm detects quiet periods using angular velocity and acceleration thresholds, then identifies active segments between them that exceed a minimum duration.

In [ ]:
print("Detecting walking bouts...")
all_bouts = detect_walking_bouts(
    left_synced.Wb, left_synced.Ab, PERIOD,
    W_threshold=W_THRESHOLD, A_threshold=A_THRESHOLD,
    min_quiet_seconds=MIN_QUIET_SECONDS, min_walk_seconds=MIN_WALK_SECONDS
)

print(f"\nDetected {len(all_bouts)} walking bouts:")
for i, bout in enumerate(all_bouts):
    bout_start = left_synced.time_datetime[bout.start_idx]
    print(f"  {i+1}. {bout_start.strftime('%H:%M:%S')} - {bout.duration_seconds:.1f}s")

## 3. Select Bouts Near Target Times

Filter the detected bouts to those occurring near the target times specified in the configuration cell. For each target, the longest bout within the search window is selected. This is useful when the recording contains multiple activities and you want to analyze a specific walk.

In [ ]:
bout_selections = []

for bout_name, target_time, search_window in WALKING_BOUT_TARGETS:
    print(f"\nSearching for '{bout_name}' near {target_time.strftime('%H:%M:%S')}...")
    matching = find_bouts_near_time(all_bouts, left_synced.time_datetime, target_time, search_window)
    
    if matching:
        # Select longest matching bout
        best = max(matching, key=lambda b: b.duration_seconds)
        start_dt = left_synced.time_datetime[best.start_idx]
        end_dt = left_synced.time_datetime[best.end_idx - 1]
        sel = BoutSelection(bout_name, best, start_dt, end_dt, confirmed=True)
        bout_selections.append(sel)
        print(f"  Found: {sel}")
    else:
        print(f"  No bouts found within {search_window}s window")

print(f"\nSelected {len(bout_selections)} bout(s) for analysis")

## 4. Process Walking Bouts

For each selected bout, slice both foot recordings and the head recording to the bout's time range. `compute_position_two_imus()` runs inertial mechanization on both feet independently, then re-detects footfalls with tuned parameters and recomputes drift-corrected velocities and positions. Each foot's strides are then segmented and step metrics (speed, length, duration) computed. Head angular velocity and acceleration statistics are also extracted.

In [ ]:
bout_results = {}

for sel in bout_selections:
    print(f"\n{'='*60}")
    print(f"Processing: {sel.name}")
    print(f"{'='*60}")
    
    result = process_walking_bout(sel.bout, left_synced, right_synced, head_synced, PERIOD)
    bout_results[sel.name] = {'selection': sel, **result}
    
    lm, rm, hm = result['left_metrics'], result['right_metrics'], result['head_metrics']
    print(f"Left:  {lm['n_steps']} steps, {lm['mean_speed']:.2f} m/s, {result['left_total_distance']:.2f} m")
    print(f"Right: {rm['n_steps']} steps, {rm['mean_speed']:.2f} m/s, {result['right_total_distance']:.2f} m")
    print(f"Head:  {hm['ang_vel_mean']:.1f} \u00b1 {hm['ang_vel_std']:.1f} deg/s")

## 5. Visualize Results

For each bout, three sets of plots are generated: (1) side-by-side stride trajectories for the left and right foot showing each step's forward vs. lateral displacement, (2) rotation-corrected 2D and 3D foot trajectories with both feet overlaid, and (3) a summary panel comparing left/right step speed and length distributions alongside head motion statistics.

In [ ]:
for name, r in bout_results.items():
    lm, rm, hm = r['left_metrics'], r['right_metrics'], r['head_metrics']
    
    # Stride trajectories
    if lm['n_steps'] > 0 and rm['n_steps'] > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        plt.sca(axes[0])
        imu.plt_ltrl_frwd_strides(r['left_strides'], show=False)
        axes[0].set_title(f'Left Foot - {name}')
        plt.sca(axes[1])
        imu.plt_ltrl_frwd_strides(r['right_strides'], show=False)
        axes[1].set_title(f'Right Foot - {name}')
        plt.tight_layout()
        plt.show()
    
    # Rotation-corrected trajectories
    plot_rotation_corrected_trajectories(r['left_walk_info'], r['right_walk_info'], title=name)
    plt.show()
    
    # Bout summary
    plot_bout_summary(name, lm, rm, hm)
    plt.show()

## 6. Summary Table

Print a compact table of all analyzed bouts with their start time, duration, step counts for each foot, mean walking speeds, and total distances walked.

In [ ]:
if bout_results:
    print(f"\n{'Bout':<15} {'Start':<10} {'Duration':>8} {'L Steps':>8} {'R Steps':>8} {'L Speed':>10} {'R Speed':>10} {'L Dist':>10} {'R Dist':>10}")
    print("-" * 100)
    for name, r in bout_results.items():
        sel = r['selection']
        lm, rm = r['left_metrics'], r['right_metrics']
        print(f"{name:<15} {sel.start_datetime.strftime('%H:%M:%S'):<10} "
              f"{sel.duration_seconds:>8.1f} {lm['n_steps']:>8} {rm['n_steps']:>8} "
              f"{lm['mean_speed']:>10.2f} {rm['mean_speed']:>10.2f} "
              f"{r['left_total_distance']:>10.2f} {r['right_total_distance']:>10.2f}")